# SetWise MM-Fit-Only Dataset EDA

This notebook audits the MM-Fit-only artifact produced by `prepare.py`:

- `prepared/setwise_mmfit_only_50hz_512.npz`
- `prepared/setwise_mmfit_only_50hz_512_metadata.json`

Whales is intentionally omitted from this v1 dataset view.

In [ ]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.style.use("default")
np.set_printoptions(precision=4, suppress=True)

In [ ]:
STEM = "setwise_mmfit_only_50hz_512"
CANDIDATE_PREPARED_DIRS = [
    Path("prepared"),
    Path("../prepared"),
    Path.cwd() / "prepared",
    Path.cwd().parent / "prepared",
    Path("/content/drive/MyDrive/SetwiseKineticDatasets/prepared"),
]

prepared_dir = next(
    (candidate for candidate in CANDIDATE_PREPARED_DIRS if (candidate / f"{STEM}.npz").exists()),
    None,
)
assert prepared_dir is not None, "Missing MM-Fit-only dataset. Run `python prepare.py --no-mount-drive` first."

npz_path = prepared_dir / f"{STEM}.npz"
metadata_path = prepared_dir / f"{STEM}_metadata.json"
assert metadata_path.exists(), f"Missing metadata: {metadata_path}"

raw = np.load(npz_path, allow_pickle=False)
with metadata_path.open("r") as f:
    metadata = json.load(f)

print(f"Loaded: {npz_path.resolve()}")
print(f"Metadata: {metadata_path.resolve()}")
print(f"Arrays: {len(raw.files)}")
print(raw.files)

In [ ]:
def as_str_array(name: str) -> np.ndarray:
    return np.asarray(raw[name]).astype(str)

X = raw["X"]
source = as_str_array("source")
split = as_str_array("split")
exercise_name = as_str_array("exercise_name")
original_exercise_name = as_str_array("original_exercise_name")
reps = raw["reps"].astype(int)
rep_supervised = raw["rep_supervised"].astype(bool)
classification_supervised = raw["classification_supervised"].astype(bool)
duration_s = raw["duration_s"].astype(float)
lengths_50hz = raw["lengths_50hz"].astype(int)
feature_names = as_str_array("feature_names")
label_names = as_str_array("label_names")

print("X shape:", X.shape)
print("dtype:", X.dtype)
print("target_hz:", float(raw["target_hz"]))
print("model_length:", int(raw["model_length"]))
print("feature_names:", feature_names.tolist())
print("label_names:", label_names.tolist())
print("classification supervised:", int(classification_supervised.sum()), "/", len(classification_supervised))
print("rep supervised:", int(rep_supervised.sum()), "/", len(rep_supervised))

In [ ]:
def counter_table(counter: Counter, headers=("Value", "Count")) -> str:
    rows = [f"| {headers[0]} | {headers[1]} |", "| --- | ---: |"]
    def key_fn(item):
        key = item[0]
        if isinstance(key, (int, np.integer)):
            return (0, int(key))
        text = str(key)
        return (0, int(text)) if text.lstrip("-").isdigit() else (1, text)
    for key, value in sorted(counter.items(), key=key_fn):
        rows.append(f"| `{key}` | {value} |")
    return "\n".join(rows)


def cross_table(row_values, col_values, row_name="row", col_name="col") -> str:
    row_values = [str(v) for v in row_values]
    col_values = [str(v) for v in col_values]
    rows = sorted(set(row_values), key=str)
    cols = sorted(set(col_values), key=str)
    counts = defaultdict(Counter)
    for r, c in zip(row_values, col_values):
        counts[r][c] += 1
    md = ["| " + row_name + " | " + " | ".join(cols) + " |", "| --- | " + " | ".join(["---:"] * len(cols)) + " |"]
    for r in rows:
        md.append("| `" + r + "` | " + " | ".join(str(counts[r][c]) for c in cols) + " |")
    return "\n".join(md)


def metric_summary(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = np.abs(y_pred - y_true)
    return {
        "mae": float(err.mean()),
        "exact": float((err == 0).mean()),
        "within_1": float((err <= 1).mean()),
        "within_2": float((err <= 2).mean()),
    }


def metrics_table(rows: dict[str, dict[str, float]]) -> str:
    md = ["| Baseline | MAE | Exact | Within 1 | Within 2 |", "| --- | ---: | ---: | ---: | ---: |"]
    for name, m in rows.items():
        md.append(f"| {name} | {m['mae']:.4f} | {m['exact']:.2%} | {m['within_1']:.2%} | {m['within_2']:.2%} |")
    return "\n".join(md)


def describe(values):
    values = np.asarray(values, dtype=float)
    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "p75": float(np.percentile(values, 75)),
        "max": float(np.max(values)),
        "mean": float(np.mean(values)),
    }

## Dataset Contract

In [ ]:
display(Markdown(counter_table(Counter(source), ("Source", "Samples"))))
display(Markdown(counter_table(Counter(split), ("Split", "Samples"))))
display(Markdown(cross_table(source, split, "source", "split")))

assert X.shape == tuple(metadata["tensor_shape"])
assert metadata["dataset"] == "mmfit"
assert metadata["source_counts"] == {"mmfit": 616}
assert int(rep_supervised.sum()) == 0
print("MM-Fit-only metadata contract checks passed.")

## Exercise Coverage

In [ ]:
display(Markdown(counter_table(Counter(exercise_name), ("Exercise", "Samples"))))
display(Markdown(cross_table(exercise_name, split, "exercise", "split")))

## Rep Label Bias and Baselines

In [ ]:
display(Markdown(counter_table(Counter(reps.tolist()), ("Reps", "Sets"))))

rows = {"Always 10": metric_summary(reps, np.full_like(reps, 10))}
display(Markdown(metrics_table(rows)))

non_10 = reps != 10
print("Non-10 sets:", int(non_10.sum()), "/", len(reps))
if non_10.any():
    display(Markdown("### Non-10 only\n" + metrics_table({"Always 10": metric_summary(reps[non_10], np.full_like(reps[non_10], 10))})))

for split_name in ["train", "val", "test"]:
    mask = split == split_name
    display(Markdown(f"### {split_name} ({int(mask.sum())} samples)\n" + metrics_table({"Always 10": metric_summary(reps[mask], np.full_like(reps[mask], 10))})))

## Durations and Lengths

In [ ]:
print("duration_s", describe(duration_s))
print("lengths_50hz", describe(lengths_50hz))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(duration_s, bins=35, alpha=0.85)
axes[0].set_title("MM-Fit set duration")
axes[0].set_xlabel("seconds")
axes[0].set_ylabel("sets")
axes[1].hist(lengths_50hz, bins=35, alpha=0.85)
axes[1].set_title("MM-Fit 50 Hz lengths before 512 resample")
axes[1].set_xlabel("timesteps")
plt.tight_layout()

## Normalization Checks

In [ ]:
train_mask = split == "train"
print("Scaler mean from metadata:", np.array(metadata["normalization"]["mean"]))
print("Scaler std from metadata:", np.array(metadata["normalization"]["std"]))
print("Train normalized channel means:", X[train_mask].reshape(-1, X.shape[-1]).mean(axis=0))
print("Train normalized channel stds:", X[train_mask].reshape(-1, X.shape[-1]).std(axis=0))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()
for i, feature in enumerate(feature_names):
    ax = axes[i]
    ax.hist(X[train_mask, :, i].ravel(), bins=60, alpha=0.8)
    ax.set_title(str(feature))
    ax.set_yscale("log")
plt.suptitle("Train-split normalized channel distributions")
plt.tight_layout()

## Example Signals

In [ ]:
def plot_sample(idx: int):
    t = np.arange(X.shape[1]) / float(raw["target_hz"])
    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    axes[0].plot(t, X[idx, :, :3])
    axes[0].set_title(f"{idx}: {exercise_name[idx]} | reps={reps[idx]} | split={split[idx]} | acc")
    axes[0].legend(feature_names[:3], ncol=3, loc="upper right")
    axes[1].plot(t, X[idx, :, 3:])
    axes[1].set_title("gyro")
    axes[1].legend(feature_names[3:], ncol=3, loc="upper right")
    axes[1].set_xlabel("resampled model time index / target_hz")
    plt.tight_layout()

for exercise in ["squats", "pushups", "jumping_jacks"]:
    candidates = np.flatnonzero(exercise_name == exercise)
    if candidates.size:
        plot_sample(int(candidates[0]))

## Readiness Checks

In [ ]:
checks = {
    "shape_is_616x512x6": X.shape == (616, 512, 6),
    "source_is_mmfit_only": set(source.tolist()) == {"mmfit"},
    "all_classification_supervised": bool(classification_supervised.all()),
    "rep_supervised_disabled": int(rep_supervised.sum()) == 0,
    "no_nan_in_X": bool(np.isfinite(X).all()),
    "label_names_are_mmfit_10": len(label_names) == 10,
}
for name, ok in checks.items():
    print(f"{name}: {ok}")
assert all(checks.values()), checks